# VisionBridge — trained model check (Colab)

**Inference/checking only.** Do not use this notebook for training. The trained `base_model.pt` is assumed to already exist.

This notebook downloads the same real sentence-level ISL-CSLTR dataset used by the VisionBridge training notebook, selects one real sentence video, extracts its actual MediaPipe Holistic keypoints using the repository extractor, and runs the trained VisionBridge model + CTC decoder on that real sample.

Training notebook is untouched. This notebook is the check-only path.


## 1 — Clone repository and make backend imports work


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_ROOT = Path('/content/VisionBridge')
if not (REPO_ROOT / 'README.md').exists():
    subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO_ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_ROOT),'pull','--ff-only'],check=True)

BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0,str(BACKEND_ROOT))
os.chdir(REPO_ROOT)

assert (BACKEND_ROOT/'app'/'__init__.py').exists()
import app
print('Repository:',REPO_ROOT)
print('Backend import root:',BACKEND_ROOT)
print('APP IMPORT: PASS')


## 2 — Install only the packages needed for the real-video check

Do not uninstall TensorFlow, Protobuf, NumPy, PyTorch, or MediaPipe. Do not run the training notebook's dependency reset.

The Colab runtime may use Python 3.13; the old `mediapipe==0.10.21` pin is not appropriate there. Current MediaPipe 0.10.35 publishes a `py3-none-manylinux_2_28_x86_64` wheel, so this check notebook uses 0.10.35 when MediaPipe is missing. citeturn366652search0


In [ ]:
import importlib.util, subprocess, sys

def ensure(module_name, package_spec):
    if importlib.util.find_spec(module_name) is None:
        print(f'Installing {package_spec} ...')
        subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir',package_spec],check=True)
        print(f'Installed {package_spec}. Restart the runtime if this was the first install of MediaPipe or NumPy.')
    else:
        print(f'{module_name}: already installed')

ensure('kagglehub','kagglehub')
ensure('pandas','pandas')
ensure('cv2','opencv-python-headless')
ensure('mediapipe','mediapipe==0.10.35')

import torch
print('Python:',sys.version.split()[0])
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))


## 3 — Download the real ISL-CSLTR sentence-level dataset

This uses the same public Kaggle dataset ID used by the existing training notebook: `drblack00/isl-csltr-indian-sign-language-dataset`. Only this check notebook downloads it; no training is performed.


In [ ]:
import glob, os
import kagglehub

dataset_path = kagglehub.dataset_download('drblack00/isl-csltr-indian-sign-language-dataset')
print('Dataset root:',dataset_path)

candidates = [
    p for p in glob.glob(os.path.join(dataset_path,'**','*Sentence_Level*'),recursive=True)
    if os.path.isdir(p) and 'Video' in os.path.basename(p)
]

if len(candidates) != 1:
    print('Sentence-level candidates:')
    for p in candidates:
        print(' ',p)
    raise RuntimeError(f'Expected exactly one sentence-level video directory, found {len(candidates)}')

VIDEO_ROOT = candidates[0]
print('VIDEO_ROOT:',VIDEO_ROOT)


## 4 — Select one real sentence video and its ground-truth label

The dataset used by the training notebook stores sentence clips under folders whose names provide the sentence label. We select a real `.mp4` and derive the displayed ground truth from its parent folder, exactly like the existing training-data preparation code.


In [ ]:
import glob
from pathlib import Path

video_files=[]
for ext in ('*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV'):
    video_files.extend(glob.glob(os.path.join(VIDEO_ROOT,'**',ext),recursive=True))
video_files=sorted(video_files)

assert video_files, f'No video files found under {VIDEO_ROOT}'
print('Real sentence videos found:',len(video_files))
print('First 10:')
for p in video_files[:10]:
    print(' ',p)

TEST_VIDEO = video_files[0]
GROUND_TRUTH = Path(TEST_VIDEO).parent.name.replace('_',' ').strip()

print('\nSelected video:',TEST_VIDEO)
print('Ground truth:',GROUND_TRUTH)


## 5 — Extract real 132-dim pose + 1404-dim face keypoints

This calls the repository's own `extract_clip_keypoints()` implementation. No synthetic arrays are used.


In [ ]:
import mediapipe as mp
import numpy as np

from backend.scripts.extract_keypoints import extract_clip_keypoints

print('Extracting real MediaPipe keypoints from the selected video...')
with mp.solutions.holistic.Holistic(static_image_mode=False,model_complexity=1) as holistic:
    pose_np, face_np = extract_clip_keypoints(TEST_VIDEO, holistic)

print('Pose:',pose_np.shape,pose_np.dtype)
print('Face:',face_np.shape,face_np.dtype)

assert pose_np.ndim == 2 and pose_np.shape[1] == 132, pose_np.shape
assert face_np.ndim == 2 and face_np.shape[1] == 1404, face_np.shape
assert pose_np.shape[0] == face_np.shape[0] and pose_np.shape[0] > 0
print('REAL KEYPOINT EXTRACTION: PASS')


## 6 — Load the already-trained model + vocabulary

If the trained artifacts are not already in the repository checkout, upload **both** files once. This does not train anything.


In [ ]:
import shutil
import torch

WEIGHTS = REPO_ROOT/'backend/app/models/weights/base_model.pt'
VOCAB = REPO_ROOT/'backend/app/models/weights/base_model.vocab.json'

if not WEIGHTS.exists() or not VOCAB.exists():
    from google.colab import files
    print('Upload BOTH base_model.pt and base_model.vocab.json')
    uploaded = files.upload()
    WEIGHTS.parent.mkdir(parents=True,exist_ok=True)
    for name in ('base_model.pt','base_model.vocab.json'):
        if name not in uploaded:
            raise FileNotFoundError(f'Missing required artifact: {name}')
        shutil.copy(name,WEIGHTS.parent/name)

from app.training.isltranslate import SimpleCharTokenizer,_downsample_to_max_length
from app.models.base_model import load_frozen_base_model,POSE_INPUT_DIM,FACE_INPUT_DIM,MAX_SEQUENCE_LENGTH
from app.services.inference_service import decode_logits

tokenizer = SimpleCharTokenizer.load(VOCAB)
state = torch.load(WEIGHTS,map_location='cpu')
assert isinstance(state,dict) and 'output_head.weight' in state
checkpoint_vocab = int(state['output_head.weight'].shape[0])
assert checkpoint_vocab == tokenizer.vocab_size, f'Checkpoint/tokenizer mismatch: {checkpoint_vocab} vs {tokenizer.vocab_size}'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_frozen_base_model(str(WEIGHTS),vocab_size=tokenizer.vocab_size).to(device).eval()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable == 0, 'Base model is not frozen.'

print('Weights:',WEIGHTS)
print('Vocab size:',tokenizer.vocab_size)
print('Device:',device)
print('Trainable parameters:',trainable)
print('CHECKPOINT VALIDATION: PASS')


## 7 — Run the real prediction

This is the actual test: real video → real MediaPipe keypoints → your trained model → the same `decode_logits()` decoder used by VisionBridge inference.


In [ ]:
pose_t = torch.from_numpy(pose_np).float()
face_t = torch.from_numpy(face_np).float()
pose_t,face_t = _downsample_to_max_length(pose_t,face_t,Path(TEST_VIDEO).stem)

assert pose_t.shape[1] == POSE_INPUT_DIM == 132
assert face_t.shape[1] == FACE_INPUT_DIM == 1404
assert pose_t.shape[0] <= MAX_SEQUENCE_LENGTH

with torch.inference_mode():
    logits = model(pose_t.unsqueeze(0).to(device),face_t.unsqueeze(0).to(device))

prediction,confidence = decode_logits(logits)

print('\n'+'='*70)
print('REAL VISIONBRIDGE MODEL CHECK')
print('='*70)
print('VIDEO:       ',Path(TEST_VIDEO).name)
print('GROUND TRUTH:',GROUND_TRUTH)
print('PREDICTED:   ',prediction)
print('CONFIDENCE:  ',round(float(confidence),4))
print('POSE INPUT:  ',tuple(pose_t.shape))
print('FACE INPUT:  ',tuple(face_t.shape))
print('LOGITS:      ',tuple(logits.shape))
print('='*70)


## 8 — Measure this prediction with CER

This is a small single-sample check, not a benchmark.


In [ ]:
def levenshtein(a,b):
    prev=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        curr=[i]+[0]*len(b)
        for j,cb in enumerate(b,1):
            curr[j]=min(prev[j]+1,curr[j-1]+1,prev[j-1]+(0 if ca==cb else 1))
        prev=curr
    return prev[-1]

truth=GROUND_TRUTH.lower().strip()
pred=prediction.lower().strip()
cer=levenshtein(pred,truth)/max(len(truth),1)

print('Ground truth:',truth)
print('Prediction:  ',pred)
print('CER:         ',round(cer,4))
print('Exact match: ',pred==truth)

if prediction and prediction != '(no sign detected)':
    print('REAL MODEL INFERENCE: PASS')
else:
    print('REAL MODEL INFERENCE: MODEL RETURNED NO TEXT')


## Final interpretation

A successful run now proves the complete inference path on a **real sign-language video**. It does not claim that one sample proves general model accuracy. For a real benchmark, run multiple held-out videos and calculate CER/WER.
